# Análise exploratória — exportações brasileiras

Exploração da fato mensal por seção carregada no PostgreSQL. As conclusões são descritivas; FOB/kg é uma razão agregada, não um preço unitário.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import psycopg
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
load_dotenv(PROJECT_ROOT / '.env')
dsn = os.environ['PG_DSN']

In [ ]:
query = '''
SELECT t.data_calendario, t.ano, t.numero_mes, s.codigo_secao, s.nome_secao,
       f.valor_fob_usd, f.quilograma_liquido
FROM fato_exportacao_mensal f
JOIN dim_tempo t USING (tempo_id)
JOIN dim_secao_produto s USING (secao_id)
ORDER BY t.data_calendario, s.codigo_secao
'''
with psycopg.connect(dsn) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        columns = [description.name for description in cursor.description]
        df = pd.DataFrame(cursor.fetchall(), columns=columns)

df['data_calendario'] = pd.to_datetime(df['data_calendario'])
df['valor_fob_usd'] = df['valor_fob_usd'].astype('float64')
df['quilograma_liquido'] = df['quilograma_liquido'].astype('float64')
df.head(), df.shape

In [ ]:
profile = {
    'linhas': len(df),
    'meses': df['data_calendario'].dt.to_period('M').nunique(),
    'secoes': df['codigo_secao'].nunique(),
    'nulos': int(df.isna().sum().sum()),
    'chaves_duplicadas': int(df.duplicated(['data_calendario', 'codigo_secao']).sum()),
}
profile, df[['valor_fob_usd', 'quilograma_liquido']].describe(percentiles=[.01, .25, .5, .75, .99])

In [ ]:
monthly_total = df.groupby(['data_calendario', 'ano', 'numero_mes'], as_index=False)['valor_fob_usd'].sum()
monthly_total = monthly_total[monthly_total['data_calendario'].dt.day == 1]
ax = monthly_total.plot(x='data_calendario', y='valor_fob_usd', figsize=(12, 5),
                        title='Valor mensal exportado (US$ FOB)', legend=False)
ax.set_xlabel('Mês')
ax.set_ylabel('US$ FOB')
plt.tight_layout()

In [ ]:
annual = df.groupby(['ano', 'codigo_secao', 'nome_secao'], as_index=False)['valor_fob_usd'].sum()
annual['participacao_fob'] = annual['valor_fob_usd'] / annual.groupby('ano')['valor_fob_usd'].transform('sum')
annual_pivot = annual.pivot(index='nome_secao', columns='ano', values='participacao_fob').fillna(0)
annual_pivot['mudanca_2022_2025_pp'] = (annual_pivot[2025] - annual_pivot[2022]) * 100
annual_pivot.sort_values('mudanca_2022_2025_pp', ascending=False)

## Interpretação

Use a tabela e os gráficos para identificar as seções com maior ganho/perda de participação. Investigue valores extremos consultando a série e o JSON bruto antes de classificá-los como erro; não os remova automaticamente. Complete a narrativa com os resultados efetivamente observados e compare as agregações com o dashboard.